In [13]:
from pyspark.sql import SparkSession
import pyspark
print(pyspark.__version__)
from pyspark.sql import SparkSession
from pyspark.sql.utils import AnalysisException
from pyspark.sql import functions as F
from pyspark.sql.types import LongType

3.4.1


In [2]:
spark6 = SparkSession.builder \
    .appName("preprocess6") \
    .master("spark://spark-master:7077") \
    .config("spark.sql.warehouse.dir", "hdfs://namenode:8020/user/nifi/stock-prices") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/22 13:33:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [10]:
df2 = spark6.read.parquet("hdfs://namenode:8020/nifi/stock-prices/stock_prices_2025-11-18-14.parquet")

In [11]:
df2.show()

+--------+----------------+----------------+--------+-----------------+----------------+----------+---------+----------------+-------------+---------+--------------------------+------+-------------------+-----------------------+----------------+--------------------+-------------------+---------------+----------------+------+-------------------+
|currency|         dayHigh|          dayLow|exchange|  fiftyDayAverage|       lastPrice|lastVolume|marketCap|            open|previousClose|quoteType|regularMarketPreviousClose|shares|tenDayAverageVolume|threeMonthAverageVolume|        timezone|twoHundredDayAverage|         yearChange|       yearHigh|         yearLow|ticker|    fetch_timestamp|
+--------+----------------+----------------+--------+-----------------+----------------+----------+---------+----------------+-------------+---------+--------------------------+------+-------------------+-----------------------+----------------+--------------------+-------------------+---------------+----

In [50]:
df2.printSchema()

root
 |-- currency: string (nullable = true)
 |-- dayHigh: double (nullable = true)
 |-- dayLow: double (nullable = true)
 |-- exchange: string (nullable = true)
 |-- fiftyDayAverage: double (nullable = true)
 |-- lastPrice: double (nullable = true)
 |-- lastVolume: integer (nullable = true)
 |-- marketCap: string (nullable = true)
 |-- open: double (nullable = true)
 |-- previousClose: double (nullable = true)
 |-- quoteType: string (nullable = true)
 |-- regularMarketPreviousClose: double (nullable = true)
 |-- shares: string (nullable = true)
 |-- tenDayAverageVolume: long (nullable = true)
 |-- threeMonthAverageVolume: long (nullable = true)
 |-- timezone: string (nullable = true)
 |-- twoHundredDayAverage: double (nullable = true)
 |-- yearChange: double (nullable = true)
 |-- yearHigh: double (nullable = true)
 |-- yearLow: double (nullable = true)
 |-- ticker: string (nullable = true)
 |-- fetch_timestamp: string (nullable = true)



In [52]:
from pyspark.sql.types import LongType
df2 = df2.withColumn("lastVolume", df2["lastVolume"].cast(LongType()))

In [53]:
df2.printSchema()

root
 |-- currency: string (nullable = true)
 |-- dayHigh: double (nullable = true)
 |-- dayLow: double (nullable = true)
 |-- exchange: string (nullable = true)
 |-- fiftyDayAverage: double (nullable = true)
 |-- lastPrice: double (nullable = true)
 |-- lastVolume: long (nullable = true)
 |-- marketCap: string (nullable = true)
 |-- open: double (nullable = true)
 |-- previousClose: double (nullable = true)
 |-- quoteType: string (nullable = true)
 |-- regularMarketPreviousClose: double (nullable = true)
 |-- shares: string (nullable = true)
 |-- tenDayAverageVolume: long (nullable = true)
 |-- threeMonthAverageVolume: long (nullable = true)
 |-- timezone: string (nullable = true)
 |-- twoHundredDayAverage: double (nullable = true)
 |-- yearChange: double (nullable = true)
 |-- yearHigh: double (nullable = true)
 |-- yearLow: double (nullable = true)
 |-- ticker: string (nullable = true)
 |-- fetch_timestamp: string (nullable = true)



# Chyba nie potrzeba preprocessingu - poprostu zapisze do csv

In [32]:
df = spark6.read.parquet("hdfs://namenode:8020/nifi/stock-prices/")

In [55]:
df = df.withColumn("lastVolume", df["lastVolume"].cast(LongType()))

In [56]:
df.printSchema()

root
 |-- currency: string (nullable = true)
 |-- dayHigh: double (nullable = true)
 |-- dayLow: double (nullable = true)
 |-- exchange: string (nullable = true)
 |-- fiftyDayAverage: double (nullable = true)
 |-- lastPrice: double (nullable = true)
 |-- lastVolume: long (nullable = true)
 |-- marketCap: string (nullable = true)
 |-- open: double (nullable = true)
 |-- previousClose: double (nullable = true)
 |-- quoteType: string (nullable = true)
 |-- regularMarketPreviousClose: double (nullable = true)
 |-- shares: string (nullable = true)
 |-- tenDayAverageVolume: long (nullable = true)
 |-- threeMonthAverageVolume: long (nullable = true)
 |-- timezone: string (nullable = true)
 |-- twoHundredDayAverage: double (nullable = true)
 |-- yearChange: double (nullable = true)
 |-- yearHigh: double (nullable = true)
 |-- yearLow: double (nullable = true)
 |-- ticker: string (nullable = true)
 |-- fetch_timestamp: string (nullable = true)



In [57]:
df.show()

+--------+----------------+----------------+--------+-----------------+----------------+----------+---------+----------------+----------------+---------+--------------------------+------+-------------------+-----------------------+----------------+--------------------+--------------------+---------------+----------------+------+-------------------+
|currency|         dayHigh|          dayLow|exchange|  fiftyDayAverage|       lastPrice|lastVolume|marketCap|            open|   previousClose|quoteType|regularMarketPreviousClose|shares|tenDayAverageVolume|threeMonthAverageVolume|        timezone|twoHundredDayAverage|          yearChange|       yearHigh|         yearLow|ticker|    fetch_timestamp|
+--------+----------------+----------------+--------+-----------------+----------------+----------+---------+----------------+----------------+---------+--------------------------+------+-------------------+-----------------------+----------------+--------------------+--------------------+--------

In [58]:
df.write \
    .option("header", "true") \
    .mode("overwrite") \
    .csv("hdfs://namenode:8020/ready_data/stock-prices")

25/11/22 14:34:50 WARN TaskSetManager: Lost task 4.0 in stage 28.0 (TID 746) (172.18.0.13 executor 0): org.apache.spark.SparkException: [TASK_WRITE_FAILED] Task failed while writing rows to hdfs://namenode:8020/ready_data/stock-prices.
	at org.apache.spark.sql.errors.QueryExecutionErrors$.taskFailedWhileWritingRowsError(QueryExecutionErrors.scala:788)
	at org.apache.spark.sql.execution.datasources.FileFormatWriter$.executeTask(FileFormatWriter.scala:420)
	at org.apache.spark.sql.execution.datasources.WriteFilesExec.$anonfun$doExecuteWrite$1(WriteFiles.scala:100)
	at org.apache.spark.rdd.RDD.$anonfun$mapPartitionsInternal$2(RDD.scala:888)
	at org.apache.spark.rdd.RDD.$anonfun$mapPartitionsInternal$2$adapted(RDD.scala:888)
	at org.apache.spark.rdd.MapPartitionsRDD.compute(MapPartitionsRDD.scala:52)
	at org.apache.spark.rdd.RDD.computeOrReadCheckpoint(RDD.scala:364)
	at org.apache.spark.rdd.RDD.iterator(RDD.scala:328)
	at org.apache.spark.scheduler.ResultTask.runTask(ResultTask.scala:92)


Py4JJavaError: An error occurred while calling o285.csv.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 4 in stage 28.0 failed 4 times, most recent failure: Lost task 4.3 in stage 28.0 (TID 750) (172.18.0.14 executor 2): org.apache.spark.SparkException: [TASK_WRITE_FAILED] Task failed while writing rows to hdfs://namenode:8020/ready_data/stock-prices.
	at org.apache.spark.sql.errors.QueryExecutionErrors$.taskFailedWhileWritingRowsError(QueryExecutionErrors.scala:788)
	at org.apache.spark.sql.execution.datasources.FileFormatWriter$.executeTask(FileFormatWriter.scala:420)
	at org.apache.spark.sql.execution.datasources.WriteFilesExec.$anonfun$doExecuteWrite$1(WriteFiles.scala:100)
	at org.apache.spark.rdd.RDD.$anonfun$mapPartitionsInternal$2(RDD.scala:888)
	at org.apache.spark.rdd.RDD.$anonfun$mapPartitionsInternal$2$adapted(RDD.scala:888)
	at org.apache.spark.rdd.MapPartitionsRDD.compute(MapPartitionsRDD.scala:52)
	at org.apache.spark.rdd.RDD.computeOrReadCheck